In [ ]:
!pip install openai Pillow requests

In [140]:
# --- REEMPLAZA TU CELDA 2 CON ESTO ---
import os
import requests
from io import BytesIO
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

# Asegúrate de tener tu archivo .env configurado
load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

MODELO_TEXTO = "google/gemini-flash-1.5:free" 

# Mantenemos los de imagen que configuramos antes
MODELO_IMAGEN_PRINCIPAL = "black-forest-labs/flux-schnell"
MODELO_IMAGEN_FALLBACK = "stabilityai/stable-diffusion-xl"

In [141]:
def generar_leyenda(prompt: str) -> str:
    """Genera el texto/leyenda con un modelo de lenguaje."""
    response = client.chat.completions.create(
        model=MODELO_TEXTO,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [142]:
# --- REEMPLAZA TU CELDA 4 CON ESTO ---
def generar_imagen(prompt_imagen: str) -> Image.Image:
    
    # Lista de modelos a intentar, en orden de preferencia
    modelos_a_intentar = [MODELO_IMAGEN_PRINCIPAL, MODELO_IMAGEN_FALLBACK]
    
    for modelo_id in modelos_a_intentar:
        try:
            print(f"📸 Intentando generar imagen con: {modelo_id}...")
            
            # --- CAMBIO CLAVE: Usamos client.images.generate ---
            response = client.images.generate(
                model=modelo_id,
                prompt=prompt_imagen,
                size="1024x1024", # Tamaño estándar
                n=1,
                quality="standard"
            )
            
            # En OpenRouter/OpenAI estándar, la URL está en data[0].url
            url_imagen = response.data[0].url
            print(f"✅ URL obtenida de {modelo_id}: {url_imagen[:60]}...")
            
            # Descargar la imagen
            img_bytes = requests.get(url_imagen, timeout=30).content
            return Image.open(BytesIO(img_bytes))
            
        except Exception as e:
            # Imprimimos el error real para diagnóstico, pero continuamos al siguiente modelo
            print(f"⚠️ Falló {modelo_id}. Error: {e}")
            continue # Intenta el siguiente modelo en la lista

    # --- Si todos los modelos fallan ---
    print("❌ Todos los modelos de imagen fallaron o están saturados.")
    # Devolvemos un cuadro GRIS (distinto al rojo anterior) para indicar fallo total
    return Image.new('RGB', (1024, 1024), color=(128, 128, 128))

In [143]:
def generacion_inicial():
    personaje = input("Introduce un personaje: ")
    escenario  = input("Introduce un escenario: ")

    prompt_leyenda = (
        f"Escribe una leyenda breve y cinematográfica (2-3 frases) "
        f"sobre '{personaje}' en '{escenario}'. "
        f"Solo el texto de la leyenda, sin títulos ni explicaciones."
    )

    prompt_imagen = (
        f"Cinematic epic illustration of {personaje} in {escenario}, "
        f"dramatic lighting, high detail, fantasy art style."
    )

    print("Generando leyenda...")
    leyenda = generar_leyenda(prompt_leyenda)
    print(f"Leyenda:\n{leyenda}")

    print("Generando imagen...")
    imagen = generar_imagen(prompt_imagen)
    imagen.save("portada_inicial.png")
    imagen.show()
    print("Imagen guardada como portada_inicial.png")

    return personaje, escenario, prompt_leyenda, prompt_imagen, leyenda

In [144]:
def iteracion_con_feedback(personaje, escenario, prompt_leyenda_original,
                           prompt_imagen_original, leyenda_original):

    feedback = input("¿Qué cambios te gustaría hacer? ")

    # Combina contexto original + feedback para mantener coherencia
    prompt_leyenda_final = (
        f"Leyenda original: {leyenda_original}\n\n"
        f"Aplica esta mejora manteniendo personaje y escenario: {feedback}\n"
        f"Devuelve solo la leyenda mejorada, sin títulos ni explicaciones."
    )

    prompt_imagen_final = (
        f"{prompt_imagen_original}, additionally: {feedback}"
    )

    print("Refinando leyenda...")
    leyenda_final = generar_leyenda(prompt_leyenda_final)
    print(f"Leyenda final:\n{leyenda_final}")

    print("Generando imagen final...")
    imagen_final = generar_imagen(prompt_imagen_final)
    imagen_final.save("portada_final.png")
    imagen_final.show()
    print("✅ Imagen final guardada como portada_final.png")

In [145]:
if __name__ == "__main__":
    try:
        print("--- 🎨 Generador de Historias e Imágenes (OpenRouter Free) ---")
        
        # 1. Ejecutar la generación inicial
        # Este método pide el personaje y escenario por consola
        p, esc, p_ley_orig, p_img_orig, ley_orig = generacion_inicial()
        
        print("\n" + "-"*30)
        
        # 2. Preguntar si se desea refinar el resultado
        continuar = input("¿Deseas mejorar la imagen o el texto? (s/n): ").lower()
        
        if continuar == 's':
            # 3. Ejecutar la iteración con feedback
            # Pasamos todos los datos recolectados en el primer paso
            iteracion_con_feedback(
                personaje=p, 
                escenario=esc, 
                prompt_leyenda_original=p_ley_orig,
                prompt_imagen_original=p_img_orig, 
                leyenda_original=ley_orig
            )
        else:
            print("¡Proceso finalizado! Revisa 'portada_inicial.png'.")

    except Exception as e:
        print(f"💥 Ocurrió un error general en el flujo: {e}")

--- 🎨 Generador de Historias e Imágenes (OpenRouter Free) ---
Generando leyenda...
💥 Ocurrió un error general en el flujo: Error code: 404 - {'error': {'message': 'No endpoints found for google/gemini-flash-1.5:free.', 'code': 404}, 'user_id': 'user_3CLwwGRyT6M37pX0W4W5X7N106v'}


In [146]:
"""
Resumen de flujo:
Usuario da personaje + escenario
        ↓
Prompt → Gemini (TEXT + IMAGE)
        ↓
Se muestra leyenda + portada_inicial.png
        ↓
Usuario da feedback ("hazlo más oscuro", "añade lluvia"...)
        ↓
Prompt original + feedback → Gemini (TEXT + IMAGE)
        ↓
Se muestra leyenda mejorada + portada_final.png
"""

'\nResumen de flujo:\nUsuario da personaje + escenario\n        ↓\nPrompt → Gemini (TEXT + IMAGE)\n        ↓\nSe muestra leyenda + portada_inicial.png\n        ↓\nUsuario da feedback ("hazlo más oscuro", "añade lluvia"...)\n        ↓\nPrompt original + feedback → Gemini (TEXT + IMAGE)\n        ↓\nSe muestra leyenda mejorada + portada_final.png\n'